# Week 02 Lab: Image Processing Through a Four-Step Pipeline

**Course:** Perceptual Computing / Computer Graphics and Human-Computer Interaction  
**Coverage:** Week 1-3 lecture material  
**Work mode:** Individual submission

## Student Information

- **Name:** _SIYYAM_KHAN_
- **Roll number:** _2k24/AIE/75_
- **GitHub username:** 2k24AIE75
- **GitHub repository:** create one **public** repository named exactly `CGHCI` on that account. Use this same repository for all labs this semester.

This lab is designed to make you practice turning image-processing rules into reliable Python programs. Complete **all five problems** and preserve your formulas, manual traces, implementation, tests, and short explanations.

## Problem 1: Thresholding a Grayscale Image

Convert each pixel to black (`0`) or white (`255`) based on a cutoff of 128. Values equal to the threshold are included in the white output. The test compares the function result with the expected 2 x 3 image.

In [13]:
import numpy as np


def threshold_image(image: np.ndarray, threshold: int) -> np.ndarray:
    """Convert a grayscale image to black and white."""
    # Pixels at or above the cutoff become white; lower values become black.
    return np.where(image >= threshold, 255, 0).astype(np.uint8)


problem_1_input = np.array(
    [[20, 128, 200],
     [100, 150, 250]],
    dtype=np.uint8
)

problem_1_expected = np.array(
    [[0, 255, 255],
     [0, 255, 255]],
    dtype=np.uint8
)

result = threshold_image(problem_1_input, 128)

# Confirm the function matches the hand-calculated threshold result.
np.testing.assert_array_equal(result, problem_1_expected)

print("Output:")
print(result)

print("Problem 1 passed")

Output:
[[  0 255 255]
 [  0 255 255]]
Problem 1 passed


## Problem 2: Estimating Image Memory

Calculate the uncompressed storage required by a 1920 x 1080 display using 24 bits per pixel. The calculation converts bits to bytes and checks the result against the expected memory size.

In [14]:
def display_memory_bytes(width: int, height: int, bpp: int) -> int:
    """Return image memory in bytes."""
    # Divide the total pixel bits by 8 to convert them to bytes.
    return width * height * bpp // 8


# Compute the expected storage for the same display dimensions and color depth.
expected_bytes = 1920 * 1080 * 24 // 8

result = display_memory_bytes(1920, 1080, 24)

# Verify the function agrees with the direct calculation.
assert result == expected_bytes

print("Memory:", result, "bytes")
print("Problem 2 passed")

Memory: 6220800 bytes
Problem 2 passed


## Problem 3: Brightness and Contrast Adjustment

Apply the linear transform $O = \alpha I + \beta$ to each pixel, clipping values to the valid 8-bit range `[0, 255]`. The sample test checks the transformed image, and the extra test verifies that an over-bright pixel is clipped to 255.

In [15]:
import numpy as np

def adjust_brightness_contrast(image: np.ndarray, alpha: float, beta: float) -> np.ndarray:
    """Apply O = alpha * I + beta and clip the result to [0, 255]."""
    # Use floating point for the transform, then clamp and restore 8-bit pixels.
    image_float = image.astype(np.float32)
    adjusted = alpha * image_float + beta
    return np.clip(adjusted, 0, 255).astype(np.uint8)

# Sample image
sample_image = np.array([
    [100, 150],
    [200, 50]
], dtype=np.uint8)

# Apply brightness and contrast adjustment
sample_output = adjust_brightness_contrast(
    sample_image,
    alpha=1.5,
    beta=-20
)

# Expected result
expected_output = np.array([
    [130, 205],
    [255, 55]
], dtype=np.uint8)

# Test
np.testing.assert_array_equal(sample_output, expected_output)

# Show output
print("Adjusted Image:")
print(sample_output)

# Extra test
extra = adjust_brightness_contrast(
    np.array([[240]], dtype=np.uint8),
    alpha=1.0,
    beta=30
)

assert extra[0, 0] == 255

print("Extra test output:")
print(extra)

print("Sample 3 passed.")

Adjusted Image:
[[130 205]
 [255  55]]
Extra test output:
[[255]]
Sample 3 passed.


## Problem 4: Contrast Stretching

Map pixel values linearly from the input interval `[in_min, in_max]` into `[out_min, out_max]`, then clip to the output limits. This example maps `[50, 150]` to `[0, 255]` and checks the midpoint as well as both endpoints.

In [16]:
import numpy as np

def contrast_stretch(
    image: np.ndarray,
    in_min: float,
    in_max: float,
    out_min: float = 0.0,
    out_max: float = 255.0,
) -> np.ndarray:
    """Change image values from one range to another."""
    # Scale relative to the input interval, then shift into the output interval.
    stretched = (image - in_min) * (out_max - out_min) / (in_max - in_min) + out_min
    return np.clip(stretched, out_min, out_max)


problem_4_input = np.array([50, 100, 150], dtype=np.float32)

expected = np.array([0.0, 127.5, 255.0], dtype=np.float32)

result = contrast_stretch(problem_4_input, 50, 150)

# Compare floating-point results with a tolerance for rounding differences.
np.testing.assert_allclose(result, expected)

print("Output:")
print(result)

print("Problem 4 passed")

Output:
[  0.  127.5 255. ]
Problem 4 passed


## Problem 5: Sobel Edge Response

Convolve the 3 x 3 image patch with horizontal and vertical Sobel kernels to obtain $G_x$ and $G_y$. Combine those responses as $\sqrt{G_x^2 + G_y^2}$ for edge strength. The vertical boundary in the sample should produce a horizontal response and no vertical response.

In [18]:
import numpy as np

def sobel_response(block: np.ndarray) -> tuple[float, float, float]:
    """Return gx, gy, and edge strength."""
    # Sobel kernels estimate intensity changes across the horizontal and vertical axes.
    gx_kernel = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1]
    ], dtype=np.float32)

    gy_kernel = np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1]
    ], dtype=np.float32)

    gx = float(np.sum(block * gx_kernel))
    gy = float(np.sum(block * gy_kernel))
    strength = float(np.sqrt(gx**2 + gy**2))

    return gx, gy, strength


problem_5_input = np.array(
    [[20, 20, 200],
     [20, 20, 200],
     [20, 20, 200]],
    dtype=np.float32
)

# Measure the sample patch's horizontal and vertical gradients.
gx, gy, strength = sobel_response(problem_5_input)

# This vertical edge has a horizontal gradient and no vertical gradient.
assert gx == 720.0 and gy == 0.0

print("Gx:", gx)
print("Gy:", gy)
print("Edge strength:", strength)
print("Problem 5 passed")

Gx: 720.0
Gy: 0.0
Edge strength: 720.0
Problem 5 passed
